In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import re

In [ ]:
MODEL_PATH="/kaggle/input/deep-past-challenge-training-gap/byt5-base-akkadian_666/"


In [ ]:
def replace_transliteration(result_str):
    result_str = result_str.replace("<gap>", "∅")
    result_str = result_str.replace("<big_gap>", "∆")
    result_str = result_str.replace("<gap>", "∅")
    result_str = result_str.replace("<big_gap>", "∆")

    result_str = result_str.replace("!","")
    result_str = result_str.replace("+","")
    result_str = result_str.replace("ı","")
    result_str = result_str.replace(":","")
    result_str = result_str.replace(";","")
    
    result_str = re.sub(r"  ",' ',result_str).strip()   
    result_str = re.sub(r"  ",' ',result_str).strip()   
    result_str = re.sub(r"  ",' ',result_str).strip()   
    
    return result_str
    

In [ ]:
def replace_translation(result_str):
    
    result_str = result_str.replace("\"","")
    result_str = result_str.replace("∅", "<gap>")
    result_str = result_str.replace("∆", "<big_gap>")
    result_str = result_str.replace("∅", "<gap>")
    result_str = result_str.replace("∆", "<big_gap>")

    return result_str
    

In [ ]:
TEST_DATA_PATH = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
BATCH_SIZE = 16
MAX_LENGTH = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(DEVICE)
model.eval()

test_df = pd.read_csv(TEST_DATA_PATH)
test_df["transliteration"]=test_df["transliteration"].apply(replace_transliteration)

In [ ]:
PREFIX = "translate Akkadian to English: "

class InferenceDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['transliteration'].astype(str).tolist()
        self.texts = [PREFIX + i for i in self.texts]
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text, 
            max_length=MAX_LENGTH, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0)
        }

test_dataset = InferenceDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Starting Inference...")
all_predictions = []

In [ ]:

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
  
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_LENGTH,
            num_beams=4,
            early_stopping=True
        )
        # outputs = model.generate(
        #     input_ids=input_ids,
        #     attention_mask=attention_mask,
        #     max_length=MAX_LENGTH,
        #     num_beams=4,
        #     do_sample=True,      # Permet de sortir de la rigidité du beam search
        #     top_p=0.95,          # Filtre les queues de distribution improbables
        #     temperature=0.85,     # Équilibre entre créativité et précision
        #     early_stopping=True
        #  )
        # outputs = model.generate(
        #     input_ids=input_ids,
        #     attention_mask=attention_mask,
        #     max_length=MAX_LENGTH,
        #     min_length=5,  # 避免输出过短（阿卡德语至少有基础语义）
        #     num_beams=6,  # 增加束搜索数，提升输出完整性（小众语种数据少，束搜索数稍高更优）
        #     early_stopping=True,
        #     #no_repeat_ngram_size=3,  # 阿卡德语转写重复多，提升到3避免重复
        #     length_penalty=1.2,  # 轻微鼓励稍长输出，避免过度截断
        #     repetition_penalty=1.1,  # 惩罚重复，减少冗余token占用长度
        #     force_words_ids=None,  
        # )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_predictions.extend([d.strip() for d in decoded])



In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": all_predictions
})
submission["translation"] = submission["translation"].apply(lambda x: x if len(x) > 0 else "broken text")
submission["translation"]=submission["translation"].apply(replace_translation)

submission.to_csv("submission.csv", index=False)
print("Submission file saved successfully!")
submission.head()

In [ ]:
# import re
# def extract_arabic_numbers(text):
#     pattern = r"(?<!\.)\d+(?:\.\d+)?"
#     numbers = re.findall(pattern, text)
    
#     return numbers

# test_df=pd.read_csv("/kaggle/input/deep-past-initiative-machine-translation/train.csv")

# end_list=[]
# for index_,t2 in enumerate(all_predictions,0):
#     append_text="... not Seal silver there and so we can pay back the merchant."

#     t1=test_df.iloc[index_]["transliteration"]


#     num_list1 = extract_arabic_numbers(t1)
#     num_list2 = extract_arabic_numbers(t2)
#     num_temp_list=[]                          
#     for num_k in num_list1:
#         if num_k not in num_list2:
#             num_temp_list.append(num_k)

  
#     append_text=" ".join(num_temp_list)+" "+append_text

#     if len(t1)-len(t2)>180:
#         #print(len)
#         t2= t2 + " "+append_text.strip()
        
#     end_list.append(t2)

# submission = pd.DataFrame({
#     "id": test_df["oare_id"],
#     "translation": end_list
# })

# submission["translation"] = submission["translation"].apply(lambda x: x if len(x) > 0 else "broken text")

# submission.to_csv("submission_extend_data_1221.csv", index=False)
# print("Submission file saved successfully!")
# submission.head()